[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day6_live.ipynb)

# Day 6 · 강의 — 내 데이터로 분류기

분류에서 탐지까지 · 남의 가중치를 받아 내 것으로

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

수업을 따라가며 진행한다.

모든 셀에 **코드가 채워져 있다.** 위에서부터 실행해 결과를 눈으로 확인한다.
강사가 설명하는 동안 값을 바꿔 가며 돌려 본다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 사진을 512칸으로 바꾼다

### 준비

아래 두 셀을 **먼저 한 번** 실행한다. 런타임을 **T4 GPU** 로 바꾼 뒤에 실행해야 뒤가 빠르다.

In [ ]:
# ── 준비 ──────────────────────────────────────────────────────────────
# 런타임 → 런타임 유형 변경 → T4 GPU 로 바꾸고 시작한다
import os, sys, time, glob, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('장치', device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

# 자료 — 드라이브 폴더를 먼저 보고, 없으면 원래 자리에서 받는다
DRIVE_ZIP = ''      # 강사가 알려 주는 파일 ID 를 넣으면 드라이브에서 받는다
if not os.path.isdir('hymenoptera_data'):
    if DRIVE_ZIP:
        os.system(f'{sys.executable} -m pip install -q gdown')
        os.system(f'gdown {DRIVE_ZIP} -O hym.zip -q')
    if not os.path.isfile('hym.zip'):
        os.system('wget -q https://download.pytorch.org/tutorial/hymenoptera_data.zip -O hym.zip')
    os.system('unzip -q hym.zip')

NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
T = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                        transforms.ToTensor(), NORM])
train = datasets.ImageFolder('hymenoptera_data/train', T)
val   = datasets.ImageFolder('hymenoptera_data/val', T)
print(train.classes, len(train), len(val))

In [ ]:
# 얼린 resnet18 으로 사진을 512칸으로 바꿔 둔다 — 한 번만 하면 된다
net = models.resnet18(weights='DEFAULT')
net.fc = nn.Identity()
net.eval().to(device)

def feats(ds):
    X, Y = [], []
    with torch.no_grad():
        for xb, yb in DataLoader(ds, batch_size=32):
            X.append(net(xb.to(device)).cpu()); Y.append(yb)
    return torch.cat(X), torch.cat(Y)

t0 = time.time()
Xtr, Ytr = feats(train)
Xva, Yva = feats(val)
print('%.1f초 · %s %s' % (time.time() - t0, tuple(Xtr.shape), tuple(Xva.shape)))

def train_head(X, Y, epochs=30, seed=42):
    torch.manual_seed(seed)
    head = nn.Linear(512, 2)
    opt = torch.optim.Adam(head.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    for e in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = perm[i:i + 32]
            opt.zero_grad(); loss_fn(head(X[b]), Y[b]).backward(); opt.step()
    return head

def acc(head, X, Y):
    with torch.no_grad():
        return (head(X).argmax(1) == Y).float().mean().item()

사진 한 장이 512칸이 된 것을 직접 본다.

In [ ]:
v = Xtr[0]
print('512칸 중 앞 12개')
print([round(float(x), 2) for x in v[:12]])
print('최소 %.2f · 최대 %.2f · 0 인 칸 %d개' % (v.min(), v.max(), int((v == 0).sum())))

plt.figure(figsize=(9, 1.4))
plt.imshow(v.reshape(8, 64), cmap='Purples', aspect='auto')
plt.yticks([]); plt.title('사진 한 장이 만든 512칸'); plt.show()

> **실습문제 1.** 얼린 특징으로 판정 층을 학습시켜 **검증 정확도**를 `a` 에 담는다.
> `train_head` 와 `acc` 는 준비 셀에 있다.

In [ ]:
head = train_head(Xtr, Ytr)
a = acc(head, Xva, Yva)

print(round(a, 4))
assert a > 0.9, f'0.9 는 넘어야 한다: {a}'

> **실습문제 2.** **종류별 정확도**를 따로 재서 `per` 에 담는다. `[개미, 벌]` 순서다.
> 정답이 `c` 인 것 중 맞힌 비율이다.

In [ ]:
pred = head(Xva).argmax(1)
per = [((pred == c) & (Yva == c)).sum().item() / (Yva == c).sum().item()
       for c in range(2)]

print([round(x, 4) for x in per])
assert len(per) == 2 and all(0 <= x <= 1 for x in per)

## 2. Threshold 를 움직여 본다

모델이 내놓는 것은 판정이 아니라 **확률**이다. 어디서 자를지는 사람이 정한다.

In [ ]:
head = train_head(Xtr, Ytr)               # seed 가 고정이라 늘 같은 결과다

with torch.no_grad():
    prob = torch.softmax(head(Xva), 1)[:, 1]      # 벌일 확률

for t in (0.2, 0.35, 0.5, 0.65, 0.8):
    p = (prob > t).long()
    a = ((p == 0) & (Yva == 0)).sum().item() / (Yva == 0).sum().item()
    b = ((p == 1) & (Yva == 1)).sum().item() / (Yva == 1).sum().item()
    print('Threshold %.2f  개미 %.3f  벌 %.3f  전체 %.4f'
          % (t, a, b, (p == Yva).float().mean()))

> **실습문제 3.** Threshold 를 **0.8** 로 올렸을 때의 전체 정확도를 `a80` 에 담는다.

In [ ]:
a80 = ((prob > 0.8).long() == Yva).float().mean().item()

print(round(a80, 4))
assert 0 < a80 <= 1

## 3. 무엇이 어디에 있는가 — 객체탐지

In [ ]:
# 탐지 — 오픈 가중치를 받아 그대로 써 본다
os.system(f'{sys.executable} -m pip install -q ultralytics')
from ultralytics import YOLO

if not os.path.isfile('bus.jpg'):
    os.system('wget -q https://ultralytics.com/images/bus.jpg')
det = YOLO('yolo11n.pt')
print('아는 종류', len(det.names), '개 · 계수',
      sum(p.numel() for p in det.model.parameters()))

학습 없이 그대로 써 본다. 박스 하나가 **이름 · 신뢰도 · 네 숫자**다.

In [ ]:
r = det('bus.jpg')[0]
for b in r.boxes:
    print('%-8s %.3f  %s' % (det.names[int(b.cls)], float(b.conf),
                             [round(v) for v in b.xyxy[0].tolist()]))

plt.figure(figsize=(5, 7))
plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off'); plt.show()

> **실습문제 4.** 신뢰도 Threshold 를 **0.05** 로 내렸을 때 박스가 몇 개인지 `n05` 에 담는다.

In [ ]:
n05 = len(det('bus.jpg', conf=0.05)[0].boxes)

print(n05)
assert n05 > len(det('bus.jpg', conf=0.25)[0].boxes)

## 4. 없는 이름을 가르친다 — 파인튜닝

가르치려면 **상자를 친 사진**이 있어야 한다. 상자는 Roboflow · Label Studio · CVAT · labelImg 같은 도구에서 그리고, `YOLO` 형식으로 Export 하면 아래 폴더가 그대로 나온다.

여기서는 이미 상자를 쳐 둔 **서명 데이터**를 받아 그 폴더를 열어 본다.

In [ ]:
from ultralytics.data.utils import check_det_dataset

info = check_det_dataset('signature.yaml')      # 없으면 받아 온다
root = str(info['path'])
print('폴더', root)
print('종류', info['names'])

for d in ('images/train', 'labels/train', 'images/val', 'labels/val'):
    print('%-14s %3d개' % (d, len(glob.glob(os.path.join(root, d, '*')))))

사진 한 장과 **같은 이름의 글자 파일**이 짝을 이룬다. 그 한 줄이 상자 하나다.

In [ ]:
p = sorted(glob.glob(root + '/images/train/*'))[0]
t = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
print('사진', os.path.basename(p))
print('라벨', open(t).read().strip())

글자 다섯 개를 **다시 상자로 되돌려** 본다. 사람이 도구에서 그린 그 상자다.

In [ ]:
import matplotlib.image as mpimg

im = mpimg.imread(p)
H, W = im.shape[:2]
c, cx, cy, w, h = map(float, open(t).read().split()[:5])

plt.figure(figsize=(6, 4))
plt.imshow(im)
plt.gca().add_patch(plt.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H),
                                  w * W, h * H, fill=False, color='#E8537A', lw=2))
plt.axis('off'); plt.title('%s · 0~1 값을 픽셀로 되돌린 상자' % info['names'][int(c)])
plt.show()

서명 데이터로 파인튜닝한다. 받아 온 그대로는 서명을 **하나도 못 찾는다**.

In [ ]:
ft = YOLO('yolo11n.pt')
ft.train(data='signature.yaml', epochs=15, imgsz=320, device=device, plots=False)
m = ft.val(data='signature.yaml')
print('mAP50 %.4f · 정밀도 %.4f · 재현율 %.4f'
      % (m.box.map50, m.box.mp, m.box.mr))
print('아는 종류', ft.names)

p = sorted(glob.glob(root + '/images/val/*'))[0]   # 검증 사진 한 장

> **실습문제 5.** 상자가 **사진 넓이의 몇 %** 를 차지하는지 `frac` 에 담는다.
> 너비와 높이는 이미 0 과 1 사이 값이다.

In [ ]:
frac = w * h * 100

print('%.1f%%' % frac)
assert 0 < frac < 100

> **실습문제 6.** 파인튜닝한 모델로 서명 사진 한 장을 보고 **찾은 박스 수**를 `nb` 에 담는다.

In [ ]:
nb = len(ft(p, conf=0.4)[0].boxes)

print(nb)
assert isinstance(nb, int)

## 5. 영상에서 같은 것을 이어 본다 — 추적

탐지만 하면 프레임마다 박스를 새로 센다. **몇 명이 지나갔는지**는 답하지 못한다.

In [ ]:
if not os.path.isfile('people.mp4'):
    os.system('wget -q https://media.roboflow.com/supervision/video-examples/'
              'people-walking.mp4 -O people.mp4')

import cv2
cap = cv2.VideoCapture('people.mp4')
boxes = 0
for _ in range(120):
    ok, fr = cap.read()
    if not ok: break
    boxes += len(det(fr, classes=[0], conf=0.35, verbose=False)[0].boxes)
cap.release()
print('탐지만 120프레임 → 박스 %d개. 그런데 몇 명인지는 모른다' % boxes)

`track` 으로 바꾸면 같은 사람에게 **같은 번호**가 붙는다.

In [ ]:
cap = cv2.VideoCapture('people.mp4')
ids = {}
for _ in range(120):
    ok, fr = cap.read()
    if not ok: break
    r = det.track(fr, classes=[0], conf=0.35, persist=True, verbose=False)[0]
    if r.boxes.id is not None:
        for i in r.boxes.id.int().tolist():
            ids[i] = ids.get(i, 0) + 1
cap.release()
print('붙은 번호 %d개' % len(ids))
print('1초(25프레임) 이상 유지된 번호 %d개' % sum(1 for v in ids.values() if v >= 25))

> **실습문제 7.** **2초 이상** 머문 사람이 몇 명인지 `long2` 에 담는다.
> 25fps 이므로 2초는 50프레임이다.

In [ ]:
long2 = sum(1 for v in ids.values() if v >= 50)

print(long2)
assert long2 <= len(ids)